# Datenstruktur & Exploration

**Ziel:** Verstehen, was wir haben, bevor wir irgendetwas verändern.

Datensatz: Amazon Book Reviews (Kaggle)  
Quelle: https://www.kaggle.com/datasets/mohamedbakhet/amazon-books-reviews/data

## Setup & Daten laden

In [ ]:
import pandas as pd
import ast
import matplotlib.pyplot as plt

df1 = pd.read_csv('../data/books_data.csv')
df2 = pd.read_csv('../data/Books_rating.csv')

df = pd.merge(df1, df2, on='Title', how='inner')
print('Daten geladen.')

---
## 1. Struktur der Daten
Wie groß ist der Datensatz? Welche Spalten gibt es, und welchen Typ haben sie?

In [ ]:
print('Zeilen / Spalten:')
print(df.shape)

In [ ]:
print('Spaltennamen:')
print(df.columns.tolist())

In [ ]:
print('Datentypen je Spalte:')
print(df.dtypes)

In [ ]:
print('Erste 10 Zeilen:')
df.head(10)

In [ ]:
print('10 zufaellige Zeilen (Ueberraschungen im Format?):')
df.sample(10, random_state=42)

---
## 2. Statistische Zusammenfassung
Min, Max, Durchschnitt, Standardabweichung – gibt es absurde Werte?

In [ ]:
print('Numerische Spalten (Min, Max, Mean, Std):')
df.describe()

In [ ]:
print('Alle Spalten inkl. Text (z.B. wie viele einzigartige Autoren, Kategorien, Publisher):')
df.describe(include='all')

---
## 3. Fehlende Werte
Welche Spalten haben Luecken, und wie viel Prozent fehlen?

In [ ]:
missing = pd.DataFrame({
    'Fehlend (absolut)': df.isnull().sum(),
    'Fehlend (%)': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Fehlend (%)', ascending=False)

missing

In [ ]:
print('Spalten mit > 50 % fehlenden Werten (vermutlich nicht brauchbar):')
missing[missing['Fehlend (%)'] > 50]

---
## 4. Zielvariable: `review/score`
Wie ist die Score-Verteilung? Gibt es Werte ausserhalb von 1–5?

In [ ]:
print('Haeufigkeit je Score:')
print(df['review/score'].value_counts().sort_index())

In [ ]:
ausreisser = df[~df['review/score'].between(1, 5)]['review/score'].value_counts()
print(f'Scores ausserhalb von 1-5: {len(ausreisser)} unterschiedliche Werte')
if len(ausreisser) > 0:
    print(ausreisser)
else:
    print('Keine Ausreisser gefunden.')

In [ ]:
counts = df['review/score'].value_counts().sort_index()
ax = counts.plot(kind='bar', color='steelblue', edgecolor='white')
ax.set_xlabel('Score')
ax.set_ylabel('Anzahl Reviews')
ax.set_title('Verteilung der Review-Scores')
ax.set_xticklabels(counts.index.astype(int), rotation=0)

total = counts.sum()
for p in ax.patches:
    pct = p.get_height() / total * 100
    ax.annotate(f'{pct:.1f}%', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
print(f'=> 5er-Anteil: {counts.get(5.0, 0)/total*100:.1f}%')

---
## 5. Duplikate
Gibt es identische Zeilen? Kann ein User dasselbe Buch mehrfach bewerten?

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Identische Zeilen: {n_dupes}')

In [ ]:
max_reviews = df.groupby(['User_id', 'Title']).size().max()
multi = df.groupby(['User_id', 'Title']).size()

print(f'Max. Bewertungen eines Users fuer dasselbe Buch: {max_reviews}')
print(f'User-Buch-Paare mit mehr als 1 Bewertung: {(multi > 1).sum()}')

---
## 6. Datentypen pruefen
Sind die Spalten so gespeichert, wie wir sie benoetigen?

In [ ]:
print('review/time – Unix-Timestamp (int) oder Datum?')
print(f'  Typ: {df["review/time"].dtype}')
sample_ts = df['review/time'].dropna().iloc[0]
print(f'  Rohwert: {sample_ts}')
print(f'  Als Datum: {pd.to_datetime(sample_ts, unit="s")}')

In [ ]:
print('Price – Zahl oder String (z.B. "$9.99")?')
print(f'  Typ: {df["Price"].dtype}')
print(f'  Beispiele: {df["Price"].dropna().head(5).tolist()}')

In [ ]:
print('review/helpfulness – Format "n/m"?')
print(f'  Typ: {df["review/helpfulness"].dtype}')
print(f'  Beispiele: {df["review/helpfulness"].dropna().head(5).tolist()}')

In [ ]:
print('categories – einzelner Wert oder Liste als String?')
sample_cats = df['categories'].dropna().head(5).tolist()
print(f'  Rohwerte: {sample_cats}')

try:
    parsed = ast.literal_eval(sample_cats[0])
    print(f'  ast.literal_eval() ergibt: {parsed} (Typ: {type(parsed).__name__})')
except Exception as e:
    print(f'  Parsing-Fehler: {e}')

---
## 7. Groessenordnungen verstehen
Wie viele einzigartige Buecher, User, Kategorien, Autoren gibt es?

In [ ]:
stats = {
    'Einzigartige Buecher (Title)':          df['Title'].nunique(),
    'Einzigartige User (User_id)':           df['User_id'].nunique(),
    'Einzigartige Kategorien (categories)':  df['categories'].nunique(),
    'Einzigartige Autoren (authors)':        df['authors'].nunique(),
    'Einzigartige Publisher':                df['publisher'].nunique(),
}

pd.Series(stats).rename('Anzahl').to_frame()

In [ ]:
print('Top-10 aktivste User (meiste Reviews):')
print(df['User_id'].value_counts().head(10).to_string())

In [ ]:
print('Top-10 meistbewertete Buecher:')
print(df['Title'].value_counts().head(10).to_string())

---
## Zusammenfassung

| Thema | Wichtigste Beobachtung |
|---|---|
| Groesse | 3 Mio. Zeilen, 19 Spalten nach Merge |
| Fehlende Werte | `Price`, `description`, `ratingsCount` haben viele NaN |
| Zielvariable | Stark schief: viele 5er, wenige 1er/2er |
| Duplikate | User kann dasselbe Buch mehrfach bewerten |
| Datentypen | `review/time` = Unix Timestamp, `categories` = Liste als String, `review/helpfulness` = String `n/m` |
| Einzigartige Werte | ~212k Buecher, ~1 Mio. User |

> **Naechste Schritte:** Datenbereinigung, Feature Engineering, Modellwahl